## 01 Persistencia y Streaming

In [ ]:
%pip install -qU langgraph langchain langchain-google-genai langchain-tavily python-dotenv aiosqlite langgraph-checkpoint-sqlite

#### Configurando la base de datos y variables de entorno

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

#### Creando el estado del agente

In [ ]:
import operator
from typing import Annotated, List, Any, Dict
from dataclasses import dataclass, field
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, BaseMessage, AnyMessage

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.checkpoint.sqlite import SqliteSaver

from typing_extensions import TypedDict

import sqlite3

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

#### Implementando la clase del agente

In [ ]:
class Agent:

    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_gemini)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)
    
    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        print(f"Mensajes enviados al modelo:", messages)
        message = self.model.invoke(messages)
        return {'messages': [message]}
    
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0
    
    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Llamando la Herramienta: {t['name']} con los siguientes argumentos: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Retornando a la LLM tras la acción!")
        return {'messages': results}

#### Configurando el modelo y herramientas

In [ ]:
# current_tavily_api_key = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY no encontrada. Asegúrate de que esté en tu .env y de que python-dotenv esté instalado.")

tool = TavilySearch(max_results=3, tavily_api_key=TAVILY_API_KEY)

prompt_system = """
Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.
Tienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).
Busca información únicamente cuando tengas certeza de qué buscar.
Si necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.
Cuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas.
"""

model = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

abot = Agent(model, [tool], system=prompt_system, checkpointer=memory)

#### Ejecutando el agente y enviando mensajes

In [ ]:
messages = [HumanMessage(content="¿Cómo está el clima en Lima - Brasil hoy (29 de diciembre de 2025)?")]
thread = {"configurable": {"thread_id": "2"}}

print(f"\n--- Pregunta 1: Clima en Lima ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")

#### Comparando el clima entre ciudades

In [ ]:
messages = [HumanMessage(content="Y el clima en Medellin?")]
thread = {"configurable": {"thread_id": "2"}}

print(f"\n--- Pregunta 2: Clima en Medellin ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")

#### Realizando comparaciones detalladas

In [ ]:
messages = [HumanMessage(content="cual ciudad está más caliente?")]
thread = {"configurable": {"thread_id": "2"}}

print(f"\n--- Pregunta 3: Comparacion ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")